# Preliminary EXIOBASE disaggregation test
========================================

Purpose: Demonstrate technical feasibility of splitting Sweden into
"Stockholm" and "Rest of Sweden" within EXIOBASE 3, then computing
production-based and consumption-based accounts for both sub-regions.

This script uses PROXY WEIGHTS (population share, rough sector estimates)
rather than the full Anthesis data collection. It is a proof-of-concept
for the April 27 workshop, not a final analysis.

Requirements:
    pip install pymrio pandas numpy matplotlib

Data:
    EXIOBASE 3 (pxp version), downloadable from Zenodo.
    The script assumes you have already downloaded and extracted it.

Usage:
    1. Set EXIOBASE_PATH to the folder containing the EXIOBASE files.
    2. Run the script: python exiobase_stockholm_test.py
    3. Results are saved to the OUTPUT_DIR folder.

Date: April 2026

In [2]:
import pymrio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import logging
import time

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

## EXIOBASE 3 download

In [1]:
import pymrio

# download EXIOBASE 3 if not already done (this will download the zip file and extract it to the specified folder).
exio3_folder = ("C:/EXIOBASE3/")
exio_downloadlog = pymrio.download_exiobase3(
    storage_folder=exio3_folder,
    system="pxp",
    years=[2024],
    doi="10.5281/zenodo.18937492" # Version 3.10.1, published Mar 23, 2026
)

print(exio_downloadlog)

Description: Download log of EXIOBASE3
MRIO Name: EXIO3
System: pxp
Version: 10.5281/zenodo.18937492
File: C:\EXIOBASE3\download_log.json
History:
20260413 18:07:08 - FILEIO -  Skip download existing file IOT_2024_pxp.zip
20260413 18:07:08 - FILEIO -  Skip download existing file IOT_2024_pxp.zip
20260413 18:07:08 - FILEIO -  Skip download existing file IOT_2024_pxp.zip
20260413 18:07:08 - FILEIO -  Downloaded https://zenodo.org/records/18937492/files/IOT_2024_pxp.zip to IOT_2024_pxp.zip
20260413 18:05:59 - NOTE -  Download log created
20260413 18:05:59 - NOTE -  python_version: 3.14.0
20260413 18:05:59 - NOTE -  pymrio_version: 0.6.3
20260413 18:05:59 - NOTE -  os: Windows
20260413 18:05:59 - NOTE -  hostname: RISEPC-WvKMUDXA
20260413 18:05:59 - NOTE -  username: rafaella


# Configuration

In [3]:
# =============================================================================
# Configuration
# =============================================================================

# Path to the extracted EXIOBASE 3 folder (adjust to your local setup).
# This should point to the folder containing the .txt files (e.g., A.txt, Y.txt).

EXIOBASE_PATH = Path("C:/EXIOBASE3/IOT_2024_pxp.zip")
if not EXIOBASE_PATH.exists():
    raise FileNotFoundError(f"EXIOBASE file not found: {EXIOBASE_PATH}")
log.info("Loading EXIOBASE from %s ...", EXIOBASE_PATH)

# Output directory for results.
OUTPUT_DIR = Path("./results_stockholm_test")
OUTPUT_DIR.mkdir(exist_ok=True)

# Sweden's country code in EXIOBASE.
SWEDEN_CODE = "SE"

# New region codes for the split.
STOCKHOLM_CODE = "SE_STHLM"
REST_OF_SWEDEN_CODE = "SE_REST"

# Base year (for labeling outputs).
BASE_YEAR = 2024

2026-04-13 19:18:45,862 INFO Loading EXIOBASE from C:\EXIOBASE3\IOT_2024_pxp.zip ...


# Step 1: Define proxy weights for Stockholm

In [4]:
# =============================================================================
# Step 1: Define proxy weights for Stockholm
# =============================================================================

def build_proxy_weights():
    """
    Build proxy sector weights for Stockholm.

    For the full analysis (Phase A of the hybrid method), these weights
    would come from Dun & Bradstreet / SCB data on sector-level input
    goods in Stockholm vs. Sweden (the w_s^Sthlm from Step 3).

    For this test, we use rough proxies based on publicly available data:
    - Stockholm County has ~24% of Sweden's population.
    - Stockholm's economy is service-heavy, construction-heavy,
      and manufacturing-light compared to the national average.

    The weights represent Stockholm's share of each sector's national output.
    They do NOT need to sum to 1 across sectors. Each weight is independent:
    it says "Stockholm accounts for X% of this sector nationally."

    These are educated guesses. Replace with actual data for the main study.
    """
    # Default weight: population share (~24%).
    default_weight = 0.24

    # Sector-specific overrides based on Stockholm's known economic structure.
    # These are applied to EXIOBASE sector names (which are long descriptive strings).
    # We use substring matching for convenience.
    overrides = {
        # Stockholm has relatively MORE of these (above population share):
        "financial": 0.45,          # Major financial hub
        "insurance": 0.40,
        "real estate": 0.35,
        "computer": 0.40,           # IT/tech hub
        "telecommunication": 0.35,
        "publishing": 0.35,
        "research": 0.35,
        "other business": 0.30,
        "public admin": 0.25,
        "education": 0.25,
        "health": 0.25,
        "construction": 0.30,       # High construction demand from population growth
        "hotel": 0.28,
        "restaurant": 0.28,
        "wholesale": 0.30,
        "retail": 0.28,
        "air transport": 0.40,      # Arlanda

        # Stockholm has relatively LESS of these (below population share):
        "mining": 0.02,             # Essentially no mining in Stockholm
        "quarrying": 0.05,
        "forestry": 0.02,           # Minimal forestry
        "logging": 0.02,
        "fishing": 0.03,
        "agriculture": 0.05,        # Very little farmland
        "cultivation": 0.05,
        "crop": 0.05,
        "cattle": 0.03,
        "pulp": 0.01,               # No pulp/paper mills in Stockholm
        "paper": 0.03,
        "wood": 0.05,               # Minimal wood manufacturing
        "basic iron": 0.01,         # No steel production
        "basic precious": 0.01,
        "basic non-ferrous": 0.02,
        "electricity": 0.15,        # Consumer, not major producer
        "steam": 0.20,
    }

    return default_weight, overrides


def assign_weights(sectors, default_weight, overrides):
    """
    Assign a Stockholm share weight to each EXIOBASE sector.

    Parameters
    ----------
    sectors : list of str
        EXIOBASE sector names (from the MultiIndex level 1).
    default_weight : float
        Default weight for sectors without a specific override.
    overrides : dict
        Substring-to-weight mapping for sector-specific overrides.

    Returns
    -------
    weights : dict
        Mapping of sector name to Stockholm share (0 to 1).
    """
    weights = {}
    for sector in sectors:
        sector_lower = sector.lower()
        matched = False
        for substring, weight in overrides.items():
            if substring in sector_lower:
                weights[sector] = weight
                matched = True
                break
        if not matched:
            weights[sector] = default_weight
    return weights


# Step 2: Load EXIOBASE

In [ ]:
# =============================================================================
# Step 2: Load EXIOBASE
# =============================================================================

def load_exiobase(path):
    """
    Load EXIOBASE 3 and parse the full system.

    Returns the pymrio IOSystem object with A, Y, and satellite extensions.
    """
    log.info(f"Loading EXIOBASE from {path} ...")
    t0 = time.time()

    exio = pymrio.parse_exiobase3(path=path)

    log.info(f"Parsed in {time.time() - t0:.1f}s. Computing A matrix ...")
    exio.calc_system()

    log.info(f"A matrix shape: {exio.A.shape}")
    log.info(f"Y matrix shape: {exio.Y.shape}")
    log.info(f"Regions: {exio.get_regions().tolist()[:5]} ... "
             f"({len(exio.get_regions())} total)")
    log.info(f"Sectors per region: {len(exio.get_sectors())}")

    # Log available extensions.
    for ext_name in ['material', 'air_emissions', 'factor_inputs']:
        ext = getattr(exio, ext_name, None)
        if ext is not None and ext.F is not None:
            log.info(f"Extension '{ext_name}': {ext.F.shape[0]} stressors")
        else:
            log.warning(f"Extension '{ext_name}': NOT FOUND")

    return exio



In [ ]:
exio3 = pymrio.parse_exiobase3(str(EXIOBASE_PATH))

In [15]:
print("Available extensions:", exio3.extensions)

Available extensions: ['Satellite Accounts_copy', 'nutrients', 'land', 'water', 'material', 'air emissions']


## Check satellite extensions

In [6]:
exio = pymrio.parse_exiobase3(path=EXIOBASE_PATH)

# Check what was actually loaded:
print("Z:", type(exio.Z), "- shape:", exio.Z.shape if exio.Z is not None else "None")
print("Y:", type(exio.Y), "- shape:", exio.Y.shape if exio.Y is not None else "None")
print("A:", exio.A is not None)
print("L:", exio.L is not None)

# Compute A from Z (and x from Z + Y):
exio.calc_system()

print("\nAfter calc_system:")
print("A:", exio.A.shape)
print("L:", exio.L.shape)

2026-04-13 19:18:59,716 INFO Read metadata from C:\EXIOBASE3\IOT_2024_pxp.zip
2026-04-13 19:18:59,718 INFO 20260413 19:18:59 - FILEIO -  Loaded IO system from C:\EXIOBASE3\IOT_2024_pxp.zip - 
2026-04-13 19:18:59,721 INFO Load data from Z.txt
2026-04-13 19:19:23,196 INFO Load data from Y.txt
2026-04-13 19:19:23,486 INFO Load data from x.txt
2026-04-13 19:19:23,499 INFO Load data from unit.txt
2026-04-13 19:19:23,528 INFO Load data from air_emissions/F.txt
2026-04-13 19:19:24,301 INFO Load data from air_emissions/F_Y.txt
2026-04-13 19:19:24,320 INFO Load data from air_emissions/unit.txt
2026-04-13 19:19:24,326 INFO 20260413 19:19:24 - FILEIO -  Added satellite account from air_emissions
2026-04-13 19:19:24,335 INFO Load data from material/F.txt
2026-04-13 19:19:24,515 INFO Load data from material/F_Y.txt
2026-04-13 19:19:24,525 INFO Load data from material/unit.txt
2026-04-13 19:19:24,530 INFO 20260413 19:19:24 - FILEIO -  Added satellite account from material
2026-04-13 19:19:24,539 INF

Z: <class 'pandas.core.frame.DataFrame'> - shape: (9800, 9800)
Y: <class 'pandas.core.frame.DataFrame'> - shape: (9800, 343)
A: False
L: False


2026-04-13 19:19:26,622 INFO 20260413 19:19:26 - MODIFICATION -  Coefficient matrix A calculated
2026-04-13 19:19:43,041 INFO 20260413 19:19:43 - MODIFICATION -  Leontief matrix L calculated



After calc_system:
A: (9800, 9800)
L: (9800, 9800)


In [7]:
# List all available extensions:
print("Extensions:", exio.get_extensions())

# Inspect what each extension contains:
for ext_name in exio.get_extensions():
    ext = getattr(exio, ext_name)
    print(f"\n{ext_name}:")
    print(f"  F: {'None' if ext.F is None else ext.F.shape}")
    if ext.F is not None:
        print(f"  First 10 rows: {ext.F.index.tolist()[:10]}")

Extensions: <generator object IOSystem.get_extensions at 0x000001FCD99586D0>

air_emissions:
  F: (420, 9800)
  First 10 rows: ['As - combustion - air', 'B(a)P - combustion - air', 'B(b)F - combustion - air', 'B(k)F - combustion - air', 'CH4 - combustion - air', 'CH4_bio - combustion - air', 'CO - combustion - air', 'CO2 - combustion - air', 'CO2_bio - combustion - air', 'Cd - combustion - air']

material:
  F: (62, 9800)
  First 10 rows: ['Domestic Extraction Used - Primary Crops - Rice', 'Domestic Extraction Used - Primary Crops - Wheat', 'Domestic Extraction Used - Primary Crops - Cereals n.e.c.', 'Domestic Extraction Used - Primary Crops - Roots and tubers', 'Domestic Extraction Used - Primary Crops - Pulses', 'Domestic Extraction Used - Primary Crops - Nuts', 'Domestic Extraction Used - Primary Crops - Vegetables', 'Domestic Extraction Used - Primary Crops - Fruits', 'Domestic Extraction Used - Primary Crops - Oil bearing crops', 'Domestic Extraction Used - Primary Crops - Sugar c

In [8]:
# Inspect all material extension rows (all 62):
mat = exio.material
for i, row_name in enumerate(mat.F.index.tolist()):
    print(f"  {i:2d}: {row_name}")

   0: Domestic Extraction Used - Primary Crops - Rice
   1: Domestic Extraction Used - Primary Crops - Wheat
   2: Domestic Extraction Used - Primary Crops - Cereals n.e.c.
   3: Domestic Extraction Used - Primary Crops - Roots and tubers
   4: Domestic Extraction Used - Primary Crops - Pulses
   5: Domestic Extraction Used - Primary Crops - Nuts
   6: Domestic Extraction Used - Primary Crops - Vegetables
   7: Domestic Extraction Used - Primary Crops - Fruits
   8: Domestic Extraction Used - Primary Crops - Oil bearing crops
   9: Domestic Extraction Used - Primary Crops - Sugar crops
  10: Domestic Extraction Used - Primary Crops - Fibres
  11: Domestic Extraction Used - Primary Crops - Spice - beverage - pharmaceutical crops
  12: Domestic Extraction Used - Primary Crops - Tobacco
  13: Domestic Extraction Used - Crop residues - Straw
  14: Domestic Extraction Used - Crop residues - Other crop residues (sugar and fodder beet leaves etc)
  15: Domestic Extraction Used - Fodder crops 

In [9]:
# Inspect GHG-relevant air emission rows:
air = exio.air_emissions
ghg_keywords = ['CO2', 'CH4', 'N2O', 'SF6', 'HFC', 'PFC']
ghg_rows = [r for r in air.F.index.tolist()
            if any(kw in r for kw in ghg_keywords)]
for r in ghg_rows:
    print(f"  {r}")

  CH4 - combustion - air
  CH4_bio - combustion - air
  CO2 - combustion - air
  CO2_bio - combustion - air
  N2O - combustion - air
  N2O_bio - combustion - air
  CH4 - non combustion - Extraction/production of (natural) gas - air
  CH4 - non combustion - Extraction/production of crude oil - air
  CH4 - non combustion - Mining of antracite - air
  CH4 - non combustion - Mining of bituminous coal - air
  CH4 - non combustion - Mining of coking coal - air
  CH4 - non combustion - Mining of lignite (brown coal) - air
  CH4 - non combustion - Mining of sub-bituminous coal - air
  CH4 - non combustion - Oil refinery - air
  CO2 - non combustion - Cement production - air
  CO2 - non combustion - Lime production - air
  SF6 - air
  HFC - air
  PFC - air
  CH4 - agriculture - air
  CO2 - agriculture - peat decay - air
  N2O - agriculture - air
  CH4 - waste - air
  CO2 - waste - biogenic - air
  CO2 - waste - fossil - air


# Step 3: Disaggregate Sweden into Stockholm + Rest of Sweden

In [ ]:
# =============================================================================
# Step 3: Disaggregate Sweden into Stockholm + Rest of Sweden
# =============================================================================

def disaggregate_sweden(exio, sector_weights):
    """
    Split Sweden into Stockholm and Rest-of-Sweden in the EXIOBASE system.

    This modifies:
    - A matrix (intermediate demand coefficients)
    - Y matrix (final demand)
    - All satellite extensions (F matrices)

    The approach:
    - Sweden's columns in A are split: Stockholm gets alpha_j * SE column,
      Rest-of-Sweden gets (1 - alpha_j) * SE column.
    - Sweden's rows in A are split similarly (proportional output split).
    - Sweden's final demand columns are split by a demand weight
      (here simplified as the average sector weight, ~24-28%).
    - Satellite F matrices are split proportionally to output.

    Parameters
    ----------
    exio : pymrio.IOSystem
        The loaded EXIOBASE system (must have A and Y computed or available).
    sector_weights : dict
        Mapping of EXIOBASE sector name to Stockholm's share (0 to 1).

    Returns
    -------
    exio_mod : pymrio.IOSystem
        Modified system with Stockholm and Rest-of-Sweden replacing Sweden.
    """
    log.info("Starting disaggregation of Sweden ...")
    t0 = time.time()

    A = exio.A.copy()
    Y = exio.Y.copy()

    # Identify Sweden's position in the MultiIndex.
    regions = A.columns.get_level_values(0)
    sectors = A.columns.get_level_values(1)
    se_mask = regions == SWEDEN_CODE

    se_sectors = A.columns[se_mask].get_level_values(1).tolist()
    log.info(f"Sweden has {sum(se_mask)} sector columns in A")

    # Build the alpha vector (Stockholm shares) for Sweden's sectors.
    alpha = pd.Series(
        [sector_weights.get(s, 0.24) for s in se_sectors],
        index=se_sectors
    )
    log.info(f"Stockholm share range: {alpha.min():.2f} - {alpha.max():.2f}, "
             f"mean: {alpha.mean():.2f}")

    # ---- Split A matrix columns (intermediate demand) ----
    # Sweden's columns become two sets: Stockholm and Rest-of-Sweden.

    se_cols = A.loc[:, se_mask]

    # Stockholm columns: scale each SE column by alpha_j.
    sthlm_cols = se_cols.copy()
    for i, sector in enumerate(se_sectors):
        sthlm_cols.iloc[:, i] = se_cols.iloc[:, i] * alpha[sector]

    # Rest-of-Sweden columns: scale by (1 - alpha_j).
    rest_cols = se_cols.copy()
    for i, sector in enumerate(se_sectors):
        rest_cols.iloc[:, i] = se_cols.iloc[:, i] * (1 - alpha[sector])

    # Rename columns.
    sthlm_col_idx = pd.MultiIndex.from_arrays(
        [[STOCKHOLM_CODE] * len(se_sectors), se_sectors],
        names=A.columns.names
    )
    rest_col_idx = pd.MultiIndex.from_arrays(
        [[REST_OF_SWEDEN_CODE] * len(se_sectors), se_sectors],
        names=A.columns.names
    )
    sthlm_cols.columns = sthlm_col_idx
    rest_cols.columns = rest_col_idx

    # ---- Split A matrix rows (output distribution) ----
    se_rows = A.loc[se_mask, :]

    sthlm_rows = se_rows.copy()
    rest_rows = se_rows.copy()
    for i, sector in enumerate(se_sectors):
        sthlm_rows.iloc[i, :] = se_rows.iloc[i, :] * alpha[sector]
        rest_rows.iloc[i, :] = se_rows.iloc[i, :] * (1 - alpha[sector])

    sthlm_row_idx = pd.MultiIndex.from_arrays(
        [[STOCKHOLM_CODE] * len(se_sectors), se_sectors],
        names=A.index.names
    )
    rest_row_idx = pd.MultiIndex.from_arrays(
        [[REST_OF_SWEDEN_CODE] * len(se_sectors), se_sectors],
        names=A.index.names
    )
    sthlm_rows.index = sthlm_row_idx
    rest_rows.index = rest_row_idx

    # ---- Reassemble A matrix ----
    # Remove Sweden, insert Stockholm and Rest-of-Sweden.
    A_no_se = A.loc[~se_mask, ~se_mask]

    # Non-SE rows, new columns.
    A_nonse_sthlm = A.loc[~se_mask, se_mask].copy()
    A_nonse_sthlm.columns = sthlm_col_idx
    A_nonse_rest = A.loc[~se_mask, se_mask].copy()
    A_nonse_rest.columns = rest_col_idx

    # For non-SE rows looking at the new Stockholm/Rest columns,
    # we need to split the original SE column purchases.
    # The A matrix coefficient a_{i,SE_j} gets split into
    # a_{i,STHLM_j} = a_{i,SE_j} (unchanged coefficient, but now it
    # represents purchases from sector j in Stockholm).
    # Actually, the coefficient itself doesn't change for the buying side.
    # The split is on the OUTPUT side (rows).
    # Let me reconsider the disaggregation logic more carefully.

    # ---- Corrected disaggregation logic ----
    #
    # The A matrix coefficient a_{ij} represents: "to produce 1 unit of
    # output in sector j, how much input from sector i is needed?"
    #
    # When we split Sweden into Stockholm (S) and Rest (R):
    #
    # COLUMNS (demand side): Each SE sector becomes two sectors.
    #   The input structure (what a sector buys) is assumed identical
    #   for Stockholm and Rest-of-Sweden. So the A column coefficients
    #   are COPIED, not scaled. What changes is the OUTPUT LEVEL (x),
    #   which is determined by final demand and the Leontief inverse.
    #
    # ROWS (supply side): Purchases from "Sweden sector i" need to be
    #   allocated between Stockholm and Rest-of-Sweden. We assume
    #   purchases from sector i in Sweden are split proportionally to
    #   each sub-region's share of that sector's output.
    #
    # So: a_{STHLM_i, j} = a_{SE_i, j} * alpha_i  (for any buyer j)
    #     a_{REST_i, j}  = a_{SE_i, j} * (1 - alpha_i)
    #
    # And: a_{i, STHLM_j} = a_{i, SE_j}  (same input recipe, copied)
    #     a_{i, REST_j}  = a_{i, SE_j}    (same input recipe, copied)
    #
    # This is the standard "proportional split" assumption.

    log.info("Rebuilding A matrix with corrected disaggregation logic ...")

    # Start fresh from the original A.
    A_orig = exio.A.copy()

    # Build new index: replace SE with SE_STHLM and SE_REST.
    new_index_tuples = []
    for region, sector in A_orig.index:
        if region == SWEDEN_CODE:
            new_index_tuples.append((STOCKHOLM_CODE, sector))
            new_index_tuples.append((REST_OF_SWEDEN_CODE, sector))
        else:
            new_index_tuples.append((region, sector))

    new_idx = pd.MultiIndex.from_tuples(new_index_tuples, names=A_orig.index.names)
    n_new = len(new_idx)

    log.info(f"New system size: {n_new} x {n_new} "
             f"(was {A_orig.shape[0]} x {A_orig.shape[1]})")

    # Allocate the new A matrix.
    A_new = pd.DataFrame(
        np.zeros((n_new, n_new)),
        index=new_idx,
        columns=new_idx
    )

    # Fill the new A matrix block by block.
    non_se_regions = [r for r in A_orig.index.get_level_values(0).unique()
                      if r != SWEDEN_CODE]

    # Block 1: non-SE to non-SE (unchanged).
    for r1 in non_se_regions:
        for r2 in non_se_regions:
            block = A_orig.loc[r1, r2]
            A_new.loc[r1, r2] = block.values

    # Block 2: SE rows to non-SE columns.
    # Original: a_{SE_i, r_j}. Split rows by alpha.
    for r2 in non_se_regions:
        block = A_orig.loc[SWEDEN_CODE, r2]  # shape: (n_sectors, n_sectors_r2)
        for i, sector_i in enumerate(se_sectors):
            a_i = alpha[sector_i]
            A_new.loc[(STOCKHOLM_CODE, sector_i), r2] = block.iloc[i, :].values * a_i
            A_new.loc[(REST_OF_SWEDEN_CODE, sector_i), r2] = block.iloc[i, :].values * (1 - a_i)

    # Block 3: non-SE rows to SE columns.
    # Original: a_{r_i, SE_j}. Columns are copied (same input recipe).
    for r1 in non_se_regions:
        block = A_orig.loc[r1, SWEDEN_CODE]  # shape: (n_sectors_r1, n_sectors)
        A_new.loc[r1, STOCKHOLM_CODE] = block.values
        A_new.loc[r1, REST_OF_SWEDEN_CODE] = block.values

    # Block 4: SE to SE (internal).
    # Original: a_{SE_i, SE_j}.
    # Row split by alpha_i, columns copied.
    block_se = A_orig.loc[SWEDEN_CODE, SWEDEN_CODE]  # (n_sec, n_sec)
    for i, sector_i in enumerate(se_sectors):
        a_i = alpha[sector_i]
        # Stockholm buys from Stockholm_i and Rest_i.
        A_new.loc[(STOCKHOLM_CODE, sector_i), STOCKHOLM_CODE] = \
            block_se.iloc[i, :].values * a_i
        A_new.loc[(REST_OF_SWEDEN_CODE, sector_i), STOCKHOLM_CODE] = \
            block_se.iloc[i, :].values * (1 - a_i)
        # Rest-of-Sweden buys from Stockholm_i and Rest_i (same recipe).
        A_new.loc[(STOCKHOLM_CODE, sector_i), REST_OF_SWEDEN_CODE] = \
            block_se.iloc[i, :].values * a_i
        A_new.loc[(REST_OF_SWEDEN_CODE, sector_i), REST_OF_SWEDEN_CODE] = \
            block_se.iloc[i, :].values * (1 - a_i)

    log.info(f"A matrix rebuilt. Shape: {A_new.shape}")

    # ---- Split Y matrix (final demand) ----
    log.info("Splitting final demand (Y) ...")

    Y_orig = exio.Y.copy()

    # Y has rows = (region, sector), columns = (region, demand_category).
    # We need to split both:
    # - SE rows (supply side): proportional to alpha.
    # - SE demand columns: proportional to demand weights.

    # For the demand split, we use a simplified approach:
    # Stockholm's share of Swedish final demand is approximated by
    # income-weighted population share. Stockholm has ~24% of population
    # but higher income, so we estimate ~28% of household demand.
    DEMAND_WEIGHT_STHLM = 0.28

    # Build new row index for Y (same as A's new index).
    Y_new_rows = new_idx

    # Build new column index for Y.
    y_col_tuples = []
    for region, cat in Y_orig.columns:
        if region == SWEDEN_CODE:
            y_col_tuples.append((STOCKHOLM_CODE, cat))
            y_col_tuples.append((REST_OF_SWEDEN_CODE, cat))
        else:
            y_col_tuples.append((region, cat))

    Y_new_cols = pd.MultiIndex.from_tuples(y_col_tuples, names=Y_orig.columns.names)

    Y_new = pd.DataFrame(
        np.zeros((len(Y_new_rows), len(Y_new_cols))),
        index=Y_new_rows,
        columns=Y_new_cols
    )

    # Fill Y: non-SE supply to non-SE demand (unchanged).
    for r_supply in non_se_regions:
        for r_demand in non_se_regions:
            if r_demand in Y_orig.columns.get_level_values(0).unique():
                try:
                    block = Y_orig.loc[r_supply, r_demand]
                    Y_new.loc[r_supply, r_demand] = block.values
                except KeyError:
                    pass

    # SE supply rows split by alpha, to non-SE demand (unchanged columns).
    for r_demand in non_se_regions:
        if r_demand in Y_orig.columns.get_level_values(0).unique():
            try:
                block = Y_orig.loc[SWEDEN_CODE, r_demand]
                for i, sector_i in enumerate(se_sectors):
                    a_i = alpha[sector_i]
                    Y_new.loc[(STOCKHOLM_CODE, sector_i), r_demand] = \
                        block.iloc[i, :].values * a_i
                    Y_new.loc[(REST_OF_SWEDEN_CODE, sector_i), r_demand] = \
                        block.iloc[i, :].values * (1 - a_i)
            except KeyError:
                pass

    # Non-SE supply to SE demand: split demand columns.
    for r_supply in non_se_regions:
        try:
            block = Y_orig.loc[r_supply, SWEDEN_CODE]
            # Stockholm demand.
            Y_new.loc[r_supply, STOCKHOLM_CODE] = block.values * DEMAND_WEIGHT_STHLM
            # Rest-of-Sweden demand.
            Y_new.loc[r_supply, REST_OF_SWEDEN_CODE] = block.values * (1 - DEMAND_WEIGHT_STHLM)
        except KeyError:
            pass

    # SE supply to SE demand: split both.
    try:
        block = Y_orig.loc[SWEDEN_CODE, SWEDEN_CODE]
        for i, sector_i in enumerate(se_sectors):
            a_i = alpha[sector_i]
            Y_new.loc[(STOCKHOLM_CODE, sector_i), STOCKHOLM_CODE] = \
                block.iloc[i, :].values * a_i * DEMAND_WEIGHT_STHLM
            Y_new.loc[(STOCKHOLM_CODE, sector_i), REST_OF_SWEDEN_CODE] = \
                block.iloc[i, :].values * a_i * (1 - DEMAND_WEIGHT_STHLM)
            Y_new.loc[(REST_OF_SWEDEN_CODE, sector_i), STOCKHOLM_CODE] = \
                block.iloc[i, :].values * (1 - a_i) * DEMAND_WEIGHT_STHLM
            Y_new.loc[(REST_OF_SWEDEN_CODE, sector_i), REST_OF_SWEDEN_CODE] = \
                block.iloc[i, :].values * (1 - a_i) * (1 - DEMAND_WEIGHT_STHLM)
        except KeyError:
            pass

    log.info(f"Y matrix rebuilt. Shape: {Y_new.shape}")

# ---- Split satellite extensions (F matrices) ----
    log.info("Splitting satellite extensions ...")

    EXTENSIONS_TO_SPLIT = ['material', 'air_emissions', 'factor_inputs',
                           'nutrients', 'land', 'water']

    satellite_F = {}
    for ext_name in EXTENSIONS_TO_SPLIT:
        ext = getattr(exio, ext_name, None)
        if ext is None or ext.F is None:
            log.warning(f"  Skipping {ext_name}: not available")
            continue

        F_orig = ext.F.copy()
        F_new = pd.DataFrame(
            np.zeros((F_orig.shape[0], n_new)),
            index=F_orig.index,
            columns=new_idx
        )

        # Non-SE columns: copy unchanged.
        for r in non_se_regions:
            F_new.loc[:, r] = F_orig.loc[:, r].values

        # SE columns: split by alpha.
        F_se = F_orig.loc[:, SWEDEN_CODE]
        for i, sector_i in enumerate(se_sectors):
            a_i = alpha[sector_i]
            F_new.loc[:, (STOCKHOLM_CODE, sector_i)] = \
                F_se.iloc[:, i].values * a_i
            F_new.loc[:, (REST_OF_SWEDEN_CODE, sector_i)] = \
                F_se.iloc[:, i].values * (1 - a_i)

        satellite_F[ext_name] = F_new
        log.info(f"  {ext_name}: split OK. Shape: {F_new.shape}")

    # ---- Rebuild the pymrio system ----
    log.info("Rebuilding pymrio IOSystem ...")
    exio_mod = pymrio.IOSystem(
        A=A_new,
        Y=Y_new,
    )

    for ext_name, F_new in satellite_F.items():
        ext = pymrio.Extension(ext_name, F=F_new)
        setattr(exio_mod, ext_name, ext)

    log.info(f"System rebuilt with {len(satellite_F)} extensions")


    return exio_mod



# Step 4: Calculate accounts

In [ ]:
# =============================================================================
# Step 4: Calculate accounts
# =============================================================================

def calculate_accounts(exio_mod):
    """
    Run the full pymrio calculation chain on the disaggregated system.

    This computes:
    - L (Leontief inverse)
    - x (total output)
    - S, M (satellite multipliers)
    - D_pba, D_cba (production-based and consumption-based accounts)
    """
    log.info("Calculating Leontief inverse and multipliers ...")
    log.info("(This may take 10-60 minutes depending on hardware and RAM)")
    t0 = time.time()

    exio_mod.calc_all()

    elapsed = time.time() - t0
    log.info(f"Calculation complete in {elapsed:.1f}s")

    return exio_mod

# Step 5: Extract and visualize results

In [ ]:
# =============================================================================
# Step 5: Extract and visualize results
# =============================================================================

def extract_results(exio_mod):
    """
    Extract Stockholm-specific results from the calculated system.

    Produces:
    - Production-based vs. consumption-based comparison for key satellites
    - GHG and material extraction breakdowns
    - Summary tables saved to CSV
    """
    results = {}

    for ext_name in ['satellite', 'impacts']:
        ext = getattr(exio_mod, ext_name, None)
        if ext is None or ext.D_cba is None:
            continue

        D_cba = ext.D_cba
        D_pba = ext.D_pba

        # ---- Stockholm's consumption-based footprint ----
        # D_cba columns are (region, demand_category).
        # Sum across demand categories for Stockholm.
        sthlm_cba = D_cba.loc[:, STOCKHOLM_CODE].sum(axis=1)

        # ---- Stockholm's production-based account ----
        # D_pba columns are (region, sector).
        # Sum across sectors for Stockholm.
        sthlm_pba = D_pba.loc[:, STOCKHOLM_CODE].sum(axis=1)

        results[ext_name] = {
            'cba': sthlm_cba,
            'pba': sthlm_pba,
            'D_cba_full': D_cba,
            'D_pba_full': D_pba,
        }

        # Save to CSV.
        comparison = pd.DataFrame({
            'Production-based (Stockholm)': sthlm_pba,
            'Consumption-based (Stockholm)': sthlm_cba,
            'Net trade (pba - cba)': sthlm_pba - sthlm_cba,
        })
        outfile = OUTPUT_DIR / f"stockholm_{ext_name}_comparison.csv"
        comparison.to_csv(outfile)
        log.info(f"Saved {ext_name} comparison to {outfile}")

    return results


def find_satellite_rows(results, keywords):
    """
    Helper: find satellite rows matching any of the given keywords.

    Parameters
    ----------
    results : dict
        Output from extract_results.
    keywords : list of str
        Substrings to match in satellite row names.

    Returns
    -------
    matching : list of str
        Matching row names.
    """
    ext_name = list(results.keys())[0]
    all_rows = results[ext_name]['cba'].index.tolist()
    matching = [r for r in all_rows
                if any(kw.lower() in r.lower() for kw in keywords)]
    return matching


def plot_comparison(results, rows, title, filename):
    """
    Bar chart comparing production-based vs consumption-based
    for selected satellite rows.
    """
    ext_name = list(results.keys())[0]
    pba_vals = results[ext_name]['pba'][rows]
    cba_vals = results[ext_name]['cba'][rows]

    # Shorten row names for display.
    short_names = [r[:50] + "..." if len(r) > 50 else r for r in rows]

    fig, ax = plt.subplots(figsize=(12, max(4, len(rows) * 0.6)))
    y_pos = np.arange(len(rows))
    bar_height = 0.35

    ax.barh(y_pos - bar_height/2, pba_vals.values, bar_height,
            label='Production-based', color='#5DCAA5', alpha=0.85)
    ax.barh(y_pos + bar_height/2, cba_vals.values, bar_height,
            label='Consumption-based', color='#AFA9EC', alpha=0.85)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(short_names, fontsize=9)
    ax.set_xlabel('Value')
    ax.set_title(title)
    ax.legend()
    ax.invert_yaxis()

    plt.tight_layout()
    outfile = OUTPUT_DIR / filename
    plt.savefig(outfile, dpi=150, bbox_inches='tight')
    plt.close()
    log.info(f"Saved plot to {outfile}")


def summarize_for_workshop(results):
    """
    Produce a high-level summary suitable for the April 27 workshop.
    """
    log.info("\n" + "=" * 60)
    log.info("WORKSHOP SUMMARY: Stockholm preliminary results")
    log.info("=" * 60)

    for ext_name, data in results.items():
        pba = data['pba']
        cba = data['cba']
        net = pba - cba

        # Find material extraction rows.
        mat_rows = [r for r in pba.index if 'extraction' in r.lower()
                    or 'domestic' in r.lower()]

        # Find GHG rows.
        ghg_rows = [r for r in pba.index if 'ghg' in r.lower()
                    or 'co2' in r.lower()
                    or 'greenhouse' in r.lower()]

        if mat_rows:
            log.info(f"\n--- Material extraction ({ext_name}) ---")
            for r in mat_rows[:10]:
                log.info(f"  {r[:60]}")
                log.info(f"    PBA: {pba[r]:>15,.0f}")
                log.info(f"    CBA: {cba[r]:>15,.0f}")
                log.info(f"    Net: {net[r]:>15,.0f} "
                         f"({'net exporter' if net[r] > 0 else 'net importer'})")

        if ghg_rows:
            log.info(f"\n--- GHG emissions ({ext_name}) ---")
            for r in ghg_rows[:5]:
                log.info(f"  {r[:60]}")
                log.info(f"    PBA: {pba[r]:>15,.0f}")
                log.info(f"    CBA: {cba[r]:>15,.0f}")
                log.info(f"    Net: {net[r]:>15,.0f}")


# Main execution

In [ ]:

# =============================================================================
# Main execution
# =============================================================================

def main():
    log.info("=" * 60)
    log.info("EXIOBASE Stockholm disaggregation test")
    log.info(f"Base year: {BASE_YEAR}")
    log.info("=" * 60)

    # Step 1: Build proxy weights.
    default_weight, overrides = build_proxy_weights()
    log.info(f"Default Stockholm share: {default_weight:.0%}")
    log.info(f"Sector-specific overrides: {len(overrides)}")

    # Step 2: Load EXIOBASE.
    exio = load_exiobase(EXIOBASE_PATH)

    # Assign weights to actual EXIOBASE sectors.
    se_sectors = exio.A.loc[SWEDEN_CODE].index.tolist()
    sector_weights = assign_weights(se_sectors, default_weight, overrides)

    # Log a few example weights.
    log.info("Example sector weights:")
    for s, w in list(sector_weights.items())[:10]:
        log.info(f"  {s[:50]:50s} -> {w:.0%}")

    # Step 3: Disaggregate.
    exio_mod = disaggregate_sweden(exio, sector_weights)

    # Free memory from original system.
    del exio

    # Step 4: Calculate.
    exio_mod = calculate_accounts(exio_mod)

    # Step 5: Extract and visualize.
    results = extract_results(exio_mod)

    # Find relevant satellite rows and plot.
    mat_keywords = ['extraction', 'domestic', 'biomass', 'metal ore',
                    'non-metallic', 'fossil']
    mat_rows = find_satellite_rows(results, mat_keywords)
    if mat_rows:
        plot_comparison(results, mat_rows[:8],
                        "Stockholm: production vs. consumption-based material extraction",
                        "stockholm_material_comparison.png")

    ghg_keywords = ['ghg', 'co2', 'greenhouse', 'GHG']
    ghg_rows = find_satellite_rows(results, ghg_keywords)
    if ghg_rows:
        plot_comparison(results, ghg_rows[:5],
                        "Stockholm: production vs. consumption-based GHG",
                        "stockholm_ghg_comparison.png")

    # Workshop summary.
    summarize_for_workshop(results)

    log.info("\n" + "=" * 60)
    log.info("Test complete. Results in: " + str(OUTPUT_DIR))
    log.info("=" * 60)


if __name__ == "__main__":
    main()